In [1]:
!nvidia-smi

Fri Sep 18 19:51:34 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os

REPO_URL = "https://github.com/siifat/green-llm-token-research"
PROJECT = "/content/drive/MyDrive/green-llm-token-research"

if not os.path.exists(PROJECT):
    !git clone "$REPO_URL" "$PROJECT"
else:
    print("Project folder already exists. Not cloning again.")

%cd "$PROJECT"

Cloning into '/content/drive/MyDrive/green-llm-token-research'...
remote: Enumerating objects: 98, done.
remote: Counting objects: 100% (98/98), done.
remote: Compressing objects: 100% (86/86), done.
remote: Total 98 (delta 4), reused 98 (delta 4), pack-reused 0 (from 0)
Receiving objects: 100% (98/98), 439.01 KiB | 1.77 MiB/s, done.
Resolving deltas: 100% (4/4), done.
/content/drive/MyDrive/green-llm-token-research


In [4]:
from pathlib import Path
import csv

csv_path = Path("green_llm_ready_bundle/green_llm_ready_100_prompts.csv")
print("Dataset exists:", csv_path.exists())

with csv_path.open("r", encoding="utf-8-sig", newline="") as f:
    rows = list(csv.DictReader(f))

print("Number of prompt pairs:", len(rows))
p3 = [r for r in rows if r["prompt_id"] == "P0003"][0]
print("P0003 source pair:", p3["source_pair_id"])

Dataset exists: True
Number of prompt pairs: 100
P0003 source pair: 1.26


In [5]:
!sudo apt-get update -qq
!sudo apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh
!ollama --version

W: https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64/InRelease: Key is stored in legacy trusted.gpg keyring (/etc/apt/trusted.gpg), see the DEPRECATION section in apt-key(8) for details.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 93 not upgraded.
Need to get 644 kB of archives.
After this operation, 1,845 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 zstd amd64 1.5.5+dfsg2-2build1.1 [644 kB]
Fetched 644 kB in 1s (445 kB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used.

In [6]:
import os, subprocess, time

os.environ["OLLAMA_MODELS"] = "/content/drive/MyDrive/ollama_models"
os.makedirs(os.environ["OLLAMA_MODELS"], exist_ok=True)

ollama_server = subprocess.Popen(
    ["ollama", "serve"],
    stdout=open("/tmp/ollama.log", "w"),
    stderr=subprocess.STDOUT,
    env=os.environ.copy()
)
time.sleep(5)
print("Ollama server started.")

Ollama server started.


In [7]:
!ollama list

NAME    ID    SIZE    MODIFIED 


In [8]:
!ollama pull glm4:9b

In [11]:
!ollama run glm4:9b "Reply with exactly: READY"

Error: could not connect to ollama server, run 'ollama serve' to start it


In [12]:
import os, subprocess, time

os.environ["OLLAMA_MODELS"] = "/content/drive/MyDrive/ollama_models"
os.makedirs(os.environ["OLLAMA_MODELS"], exist_ok=True)

ollama_server = subprocess.Popen(
    ["ollama", "serve"],
    stdout=open("/tmp/ollama.log", "w"),
    stderr=subprocess.STDOUT,
    env=os.environ.copy()
)
time.sleep(5)
print("Ollama server started.")

Ollama server started.


In [13]:
!ollama list

NAME       ID              SIZE      MODIFIED      
glm4:9b    5b699761eca5    5.5 GB    2 minutes ago    


In [14]:
!ollama run glm4:9b "Reply with exactly: READY"


READY



In [15]:
!ollama ps

NAME       ID              SIZE      PROCESSOR    CONTEXT    UNTIL              
glm4:9b    5b699761eca5    5.4 GB    100% GPU     4096       4 minutes from now    


In [16]:
from pathlib import Path

source = Path("scripts/run_experiment.py")
target = Path("scripts/run_experiment_glm4_colab.py")

text = source.read_text(encoding="utf-8")
text = text.replace('MODEL = "gemma3:4b"', 'MODEL = "glm4:9b"')
text = text.replace(
    'output_path = project_root / "data" / "raw" / "runs.jsonl"',
    'output_path = project_root / "data" / "raw" / "runs_glm4_9b_colab.jsonl"'
)
text = text.replace(
    'error_path = project_root / "logs" / "experiment_errors.jsonl"',
    'error_path = project_root / "logs" / "experiment_errors_glm4_9b_colab.jsonl"'
)
target.write_text(text, encoding="utf-8")
print("Created:", target)

Created: scripts/run_experiment_glm4_colab.py


In [17]:
!grep -n 'MODEL =' scripts/run_experiment_glm4_colab.py

11:MODEL = "glm4:9b"


In [18]:
!python scripts/run_experiment_glm4_colab.py --limit 1

Model: glm4:9b
Tasks in scope: 1
Expected runs: 2
Already completed: 0
Remaining: 2
Output: /content/drive/MyDrive/green-llm-token-research/data/raw/runs_glm4_9b_colab.jsonl

Warming up the model...
Warm-up complete.

[1/1] P0001 - baseline ... done | in=88 out=228 total=316 time=6.54s
[1/1] P0001 - optimized ... done | in=56 out=216 total=272 time=6.21s

Run pass finished.
Completed: 2/2
Still missing: 0

All experimental runs completed successfully.


In [19]:
from pathlib import Path

for p in [
    Path("data/raw/runs_glm4_9b_colab.jsonl"),
    Path("logs/experiment_errors_glm4_9b_colab.jsonl"),
]:
    if p.exists():
        p.unlink()
        print("Deleted pilot file:", p)

Deleted pilot file: data/raw/runs_glm4_9b_colab.jsonl


In [20]:
!python scripts/run_experiment_glm4_colab.py

Model: glm4:9b
Tasks in scope: 100
Expected runs: 200
Already completed: 0
Remaining: 200
Output: /content/drive/MyDrive/green-llm-token-research/data/raw/runs_glm4_9b_colab.jsonl

Warming up the model...
Warm-up complete.

[1/100] P0001 - baseline ... done | in=88 out=228 total=316 time=6.70s
[1/100] P0001 - optimized ... done | in=56 out=216 total=272 time=6.36s
[2/100] P0002 - optimized ... done | in=70 out=674 total=744 time=20.30s
[2/100] P0002 - baseline ... done | in=85 out=622 total=707 time=19.27s
[3/100] P0003 - baseline ... done | in=83 out=404 total=487 time=13.09s
[3/100] P0003 - optimized ... done | in=62 out=260 total=322 time=8.62s
[4/100] P0004 - optimized ... done | in=48 out=169 total=217 time=5.60s
[4/100] P0004 - baseline ... done | in=69 out=176 total=245 time=5.91s
[5/100] P0005 - baseline ... done | in=68 out=143 total=211 time=4.87s
[5/100] P0005 - optimized ... done | in=56 out=147 total=203 time=5.03s
[6/100] P0006 - optimized ... done | in=61 out=332 total=3

In [21]:
import json
from pathlib import Path

result_file = Path("data/raw/runs_glm4_9b_colab.jsonl")

records = [
    json.loads(line)
    for line in result_file.read_text(encoding="utf-8").splitlines()
    if line.strip()
]

print("Total records:", len(records))

Total records: 200


In [22]:
from collections import Counter

print(Counter(r["variant"] for r in records))

Counter({'baseline': 100, 'optimized': 100})


In [23]:
from collections import Counter

keys = [(r["prompt_id"], r["variant"]) for r in records]
counts = Counter(keys)

bad = {k: v for k, v in counts.items() if v != 1}

print("Unique prompt/variant records:", len(counts))
print("Problems:", bad)

Unique prompt/variant records: 200
Problems: {}


In [24]:
from collections import defaultdict

pairs = defaultdict(dict)

for r in records:
    pairs[r["prompt_id"]][r["variant"]] = r

baseline_input = sum(p["baseline"]["input_tokens"] for p in pairs.values())
optimized_input = sum(p["optimized"]["input_tokens"] for p in pairs.values())

baseline_output = sum(p["baseline"]["output_tokens"] for p in pairs.values())
optimized_output = sum(p["optimized"]["output_tokens"] for p in pairs.values())

baseline_total = sum(p["baseline"]["total_tokens"] for p in pairs.values())
optimized_total = sum(p["optimized"]["total_tokens"] for p in pairs.values())

baseline_time = sum(p["baseline"]["wall_time_s"] for p in pairs.values())
optimized_time = sum(p["optimized"]["wall_time_s"] for p in pairs.values())

def reduction(b, o):
    return (b - o) / b * 100

fewer_total = sum(
    p["optimized"]["total_tokens"] < p["baseline"]["total_tokens"]
    for p in pairs.values()
)

faster = sum(
    p["optimized"]["wall_time_s"] < p["baseline"]["wall_time_s"]
    for p in pairs.values()
)

print("=== GLM-4 9B SUMMARY ===")
print(f"Input-token reduction: {reduction(baseline_input, optimized_input):.2f}%")
print(f"Output-token reduction: {reduction(baseline_output, optimized_output):.2f}%")
print(f"Total-token reduction: {reduction(baseline_total, optimized_total):.2f}%")
print(f"Wall-time reduction: {reduction(baseline_time, optimized_time):.2f}%")
print(f"Optimized used fewer total tokens in: {fewer_total}/100 pairs")
print(f"Optimized was faster in: {faster}/100 pairs")

=== GLM-4 9B SUMMARY ===
Input-token reduction: 57.87%
Output-token reduction: 14.12%
Total-token reduction: 28.19%
Wall-time reduction: 14.82%
Optimized used fewer total tokens in: 81/100 pairs
Optimized was faster in: 77/100 pairs


In [25]:
from collections import defaultdict
import statistics

pairs = defaultdict(dict)
for r in records:
    pairs[r["prompt_id"]][r["variant"]] = r

baseline_input = sum(p["baseline"]["input_tokens"] for p in pairs.values())
optimized_input = sum(p["optimized"]["input_tokens"] for p in pairs.values())

baseline_output = sum(p["baseline"]["output_tokens"] for p in pairs.values())
optimized_output = sum(p["optimized"]["output_tokens"] for p in pairs.values())

baseline_total = sum(p["baseline"]["total_tokens"] for p in pairs.values())
optimized_total = sum(p["optimized"]["total_tokens"] for p in pairs.values())

baseline_time = sum(p["baseline"]["wall_time_s"] for p in pairs.values())
optimized_time = sum(p["optimized"]["wall_time_s"] for p in pairs.values())

def reduction(b, o):
    return (b - o) / b * 100

fewer_input = sum(p["optimized"]["input_tokens"] < p["baseline"]["input_tokens"] for p in pairs.values())
fewer_output = sum(p["optimized"]["output_tokens"] < p["baseline"]["output_tokens"] for p in pairs.values())
fewer_total = sum(p["optimized"]["total_tokens"] < p["baseline"]["total_tokens"] for p in pairs.values())
faster = sum(p["optimized"]["wall_time_s"] < p["baseline"]["wall_time_s"] for p in pairs.values())

pair_reductions = [
    reduction(p["baseline"]["total_tokens"], p["optimized"]["total_tokens"])
    for p in pairs.values()
]
mean_reduction = statistics.mean(pair_reductions)
median_reduction = statistics.median(pair_reductions)

print("=== GLM-4 9B SUMMARY ===")
print(f"Prompt pairs: {len(pairs)}\n")
print(f"Input tokens:  baseline={baseline_input:,}  optimized={optimized_input:,}  reduction={reduction(baseline_input, optimized_input):.2f}%")
print(f"Output tokens: baseline={baseline_output:,}  optimized={optimized_output:,}  reduction={reduction(baseline_output, optimized_output):.2f}%")
print(f"Total tokens:  baseline={baseline_total:,}  optimized={optimized_total:,}  reduction={reduction(baseline_total, optimized_total):.2f}%\n")
print(f"Wall time:     baseline={baseline_time:.2f}s  optimized={optimized_time:.2f}s  reduction={reduction(baseline_time, optimized_time):.2f}%\n")
print(f"Optimized used fewer input tokens in:  {fewer_input}/100 pairs")
print(f"Optimized used fewer output tokens in: {fewer_output}/100 pairs")
print(f"Optimized used fewer total tokens in:  {fewer_total}/100 pairs")
print(f"Optimized was faster in:               {faster}/100 pairs\n")
print(f"Mean pair-level total-token reduction:   {mean_reduction:.2f}%")
print(f"Median pair-level total-token reduction: {median_reduction:.2f}%\n")
print(f"PASS: Summary calculated from all {len(pairs)} prompt pairs.")

=== GLM-4 9B SUMMARY ===
Prompt pairs: 100

Input tokens:  baseline=23,378  optimized=9,850  reduction=57.87%
Output tokens: baseline=49,341  optimized=42,372  reduction=14.12%
Total tokens:  baseline=72,719  optimized=52,222  reduction=28.19%

Wall time:     baseline=1761.70s  optimized=1500.62s  reduction=14.82%

Optimized used fewer input tokens in:  100/100 pairs
Optimized used fewer output tokens in: 76/100 pairs
Optimized used fewer total tokens in:  81/100 pairs
Optimized was faster in:               77/100 pairs

Mean pair-level total-token reduction:   23.49%
Median pair-level total-token reduction: 22.75%

PASS: Summary calculated from all 100 prompt pairs.
